# Module 1: Statistics for Quant Finance

This notebook teaches you the statistical properties of financial returns —
the concepts that show up every day on a risk desk.

**How to use:** Run each cell with Shift+Enter. Read the explanations. Change the ticker and re-run to build intuition.

---

In [ ]:
# Setup — run this first
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from data_loader import fetch_prices, fetch_multi_prices

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
TRADING_DAYS = 252
print("Ready.")

## 1. Fetch Data & Compute Returns

Change  below to explore any asset. Try your fund's benchmark.

In [ ]:
# ============================
# CHANGE THESE
# ============================
TICKER = "SPY"
START  = "2018-01-01"
END    = "2025-12-31"

prices = fetch_prices(TICKER, START, END)
returns = prices.pct_change().dropna()

print(f"Loaded {len(prices)} days of {TICKER}")
print(f"Date range: {prices.index[0].date()} to {prices.index[-1].date()}")
prices.head()

### How returns are computed

A **simple return** measures the percentage change from one day to the next:

938r_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1938

For example, if SPY goes from ( to ,.50:

938r = \frac{454.50 - 450}{450} = 0.01 = 1\%938

There's also the **log return**: ^{\log} = \ln\left(\frac{P_t}{P_{t-1}}\right)$

- Simple returns are **additive across assets** (useful for portfolio PnL)
- Log returns are **additive across time** (useful for compounding math)
- For small returns, they're nearly identical

In [ ]:
# Show a few returns so you can see the actual numbers
sample = pd.DataFrame({
    "Price": prices.head(6),
    "Simple Return": returns.head(5),
    "Log Return": np.log(prices / prices.shift(1)).dropna().head(5)
})
sample

---
## 2. Volatility

Volatility measures how much returns **spread out** from the mean. It's the single most important number on a risk report.

### The formula

Daily volatility is the **standard deviation** of daily returns:

938\sigma_{\text{daily}} = \sqrt{\frac{1}{N-1} \sum_{t=1}^{N} (r_t - \bar{r})^2}938

where $\bar{r}$ is the mean daily return and $ is the number of observations.

### Annualising volatility

**Key insight:** variance scales linearly with time, so volatility scales with $\sqrt{\text{time}}$:

938\sigma_{\text{annual}} = \sigma_{\text{daily}} \times \sqrt{252}938

Why 252? That's the number of trading days in a year.

**Why sqrt?** If daily variance is $, then over 252 independent days the total variance is 52V$. Volatility is $\sqrt{\text{variance}}$, so:

938\sigma_{\text{annual}} = \sqrt{252 \cdot V} = \sqrt{V} \cdot \sqrt{252} = \sigma_{\text{daily}} \cdot \sqrt{252}938

In [ ]:
# Build volatility step by step
daily_vol = returns.std()
annual_vol = daily_vol * np.sqrt(TRADING_DAYS)

print(f"Step 1 — Daily std dev:    {daily_vol:.4%}")
print(f"Step 2 — × sqrt(252):      × {np.sqrt(252):.2f}")
print(f"Step 3 — Annual volatility: {annual_vol:.2%}")
print()
print(f"What this means:")
print(f"  On a typical day, {TICKER} moves about ±{daily_vol:.2%}")
print(f"  Over a year, {TICKER} typically moves ±{annual_vol:.1%} (1 std dev)")
print(f"  95% of the time: ±{2*annual_vol:.1%} (2 std devs)")

In [ ]:
# Visualise: price chart + returns with ±1 std dev lines
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.plot(prices, linewidth=0.8)
ax1.set_title(f"{TICKER} — Price")
ax1.set_ylabel("Price ($)")

ax2.bar(returns.index, returns.values, width=1, alpha=0.5, color="steelblue")
ax2.axhline(0, color="grey", linewidth=0.5)
ax2.axhline(daily_vol, color="red", ls="--", lw=0.8, label=f"+1σ ({daily_vol:.2%})")
ax2.axhline(-daily_vol, color="red", ls="--", lw=0.8, label=f"-1σ ({-daily_vol:.2%})")
ax2.set_title(f"{TICKER} — Daily Returns")
ax2.set_ylabel("Daily Return")
ax2.legend()
plt.tight_layout()
plt.show()

print("Notice: returns CLUSTER — calm periods then volatile periods.")
print("This is called "volatility clustering" and it's why GARCH models exist.")

---
## 3. Descriptive Statistics — The Risk Report One-Pager

These are the numbers you see on every risk report at work.

In [ ]:
stats_table = {
    "Mean daily return":  f"{returns.mean():.4%}",
    "Mean annual return": f"{returns.mean() * 252:.2%}",
    "Daily volatility":   f"{returns.std():.4%}",
    "Annual volatility":  f"{returns.std() * np.sqrt(252):.2%}",
    "Skewness":           f"{returns.skew():.2f}",
    "Excess kurtosis":    f"{returns.kurtosis():.2f}",
    "Worst day":          f"{returns.min():.2%}",
    "Best day":           f"{returns.max():.2%}",
    "Median daily":       f"{returns.median():.4%}",
    "% positive days":    f"{(returns > 0).mean():.1%}",
}

print(f"  Return Statistics: {TICKER}")
print(f"  {"="*45}")
for k, v in stats_table.items():
    print(f"  {k:<22}: {v:>10}")

---
## 4. Skewness — Are Crashes Bigger Than Rallies?

### The formula

938\text{Skewness} = \frac{1}{N} \sum_{t=1}^{N} \left( \frac{r_t - \bar{r}}{\sigma} \right)^3938

- **Skew = 0** → symmetric (normal distribution)
- **Skew < 0** → left tail is longer → big down days are bigger than big up days
- **Skew > 0** → right tail is longer

### Why negative skew matters

Most equities have negative skew. This means:
- Crashes are **sharper and faster** than rallies
- March 2020: SPY fell ~34% in 23 trading days. Recovery took ~5 months.
- Your risk models that assume symmetry will **underestimate** crash probability
- This is why your fund has stop-losses and drawdown limits

In [ ]:
skew = returns.skew()

print(f"Skewness: {skew:.2f}  (normal = 0)")
print()
if skew < 0:
    print(f"→ NEGATIVE skew: crashes are sharper than rallies.")
    print(f"  Worst day:  {returns.min():.2%}")
    print(f"  Best day:   {returns.max():.2%}")
    print(f"  The worst day is {abs(returns.min()/returns.max()):.1f}x bigger than the best day.")
else:
    print(f"→ Positive skew: unusual for equities.")

---
## 5. Kurtosis (Fat Tails) — Extreme Days Happen More Than You Think

### The formula

938\text{Excess Kurtosis} = \frac{1}{N} \sum_{t=1}^{N} \left( \frac{r_t - \bar{r}}{\sigma} \right)^4 - 3938

We subtract 3 because a normal distribution has kurtosis = 3 (so "excess" kurtosis = 0 for normal).

- **Kurtosis = 0** → normal-sized tails
- **Kurtosis > 0** → **fat tails** → extreme events happen more often than normal predicts

### The practical impact

| Event | Normal distribution says | Reality |
|-------|------------------------|--------|
| 3-sigma day (e.g. -4.5%) | Once every 1.5 years | Several times per year |
| 4-sigma day (e.g. -6%) | Once every 126 years | Every few years |
| 5-sigma day (e.g. -7.5%) | Once every 13,000 years | Has happened multiple times since 2000 |

This is why LTCM blew up — their models called their losses a "10-sigma event" (impossible under normal assumptions). With fat tails, it was more like 3-4 sigma (rare but very possible).

In [ ]:
kurt = returns.kurtosis()
sigma = returns.std()

print(f"Excess kurtosis: {kurt:.2f}  (normal = 0)")
print()

# Count extreme days vs what normal predicts
for n_sigma in [2, 3, 4]:
    actual = (returns.abs() > n_sigma * sigma).sum()
    expected = 2 * stats.norm.sf(n_sigma) * len(returns)
    ratio = actual / max(expected, 0.1)
    print(f"  {n_sigma}-sigma days:  actual = {actual:>3d},  normal predicts = {expected:>5.1f},  ratio = {ratio:.1f}x")

print()
print("→ Fat tails mean your VaR is WRONG if it assumes normality.")
print("  Use historical VaR or fit a Student-t distribution instead.")

### Visualising fat tails

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram vs normal
mu, sigma = returns.mean(), returns.std()
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
normal_pdf = stats.norm.pdf(x, mu, sigma)

ax1.hist(returns, bins=100, density=True, alpha=0.6, color="steelblue", label="Actual returns")
ax1.plot(x, normal_pdf, "r-", linewidth=2, label="Normal (theoretical)")
ax1.set_title(f"{TICKER} — Actual vs Normal")
ax1.set_xlabel("Daily Return")
ax1.set_ylabel("Density")
ax1.legend()

# Right: QQ plot
stats.probplot(returns.dropna(), dist="norm", plot=ax2)
ax2.set_title(f"{TICKER} — QQ Plot")
ax2.get_lines()[0].set_markerfacecolor("steelblue")
ax2.get_lines()[0].set_markersize(3)

plt.tight_layout()
plt.show()

print("Left:  The tails of actual returns extend FURTHER than the red normal curve.")
print("Right: If returns were normal, dots would sit on the red line.")
print("       Dots curving away at the edges = fat tails.")

---
## 6. Jarque-Bera Test — Is It Normal?

### The formula

938JB = \frac{N}{6} \left( S^2 + \frac{K^2}{4} \right)938

where $ = skewness, $ = excess kurtosis, $ = sample size.

Under the null hypothesis (returns ARE normal), $ follows a chi-squared distribution with 2 degrees of freedom.

### How to read the p-value

The p-value answers: **"If returns were truly normal, what is the probability of seeing skewness and kurtosis this extreme?"**

| p-value | Interpretation |
|---------|---------------|
| p > 0.05 | Can't reject normality (returns *might* be normal) |
| p < 0.05 | Reject normality at 95% confidence |
| p < 0.01 | Strong evidence against normality |
| p < 0.001 | Overwhelming evidence — definitely not normal |

For financial returns, p is almost always < 0.001.

In [ ]:
jb_stat, jb_p = stats.jarque_bera(returns.dropna())

print(f"Jarque-Bera statistic: {jb_stat:.1f}")
print(f"p-value:              {jb_p:.8f}")
print()

if jb_p < 0.001:
    print(f"p = {jb_p:.8f}")
    print(f"→ There is a {jb_p*100:.4f}% chance that truly normal data")
    print(f"  would look this extreme. That's essentially zero.")
    print(f"  Verdict: DEFINITELY not normal.")
elif jb_p < 0.05:
    print(f"→ NOT normal (p < 0.05).")
else:
    print(f"→ Cannot reject normality (unusual for financial returns).")

print()
print("WHY THIS MATTERS:")
print("  Parametric VaR assumes normality → it UNDERESTIMATES tail risk.")
print("  Use historical VaR or Monte Carlo with a fat-tailed distribution.")

---
## 7. Correlation — What Moves Together?

### The formula

938\rho_{X,Y} = \frac{\text{Cov}(X, Y)}{\sigma_X \cdot \sigma_Y} = \frac{\sum (r_X - \bar{r}_X)(r_Y - \bar{r}_Y)}{\sqrt{\sum (r_X - \bar{r}_X)^2} \cdot \sqrt{\sum (r_Y - \bar{r}_Y)^2}}938

- **ρ = +1** → move perfectly together
- **ρ = 0** → no linear relationship
- **ρ = -1** → move perfectly opposite

### Why it matters at a fund

- **Low correlation = diversification.** A PM with uncorrelated books can run more gross.
- **Correlation spikes in crises.** The diversification you thought you had can disappear when you need it most (March 2020: everything sold off together).
- **Negative correlation = hedging.** SPY + TLT historically move opposite — this is the foundation of 60/40 portfolios.

In [ ]:
tickers = ["SPY", "TLT", "GLD", "QQQ", "XLE"]
multi_prices = fetch_multi_prices(tickers, START, END)
multi_returns = multi_prices.pct_change().dropna()

corr = multi_returns.corr()
print("Correlation Matrix:")
print(corr.round(2).to_string())

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(tickers)))
ax.set_yticks(range(len(tickers)))
ax.set_xticklabels(tickers, rotation=45)
ax.set_yticklabels(tickers)

for i in range(len(tickers)):
    for j in range(len(tickers)):
        ax.text(j, i, f"{corr.values[i,j]:.2f}",
                ha="center", va="center", fontsize=11,
                color="white" if abs(corr.values[i,j]) > 0.5 else "black")

plt.colorbar(im)
ax.set_title("Return Correlation Heatmap")
plt.tight_layout()
plt.show()

print(f"SPY vs TLT: {corr.loc["SPY","TLT"]:.2f} → stocks and bonds move opposite")
print(f"SPY vs QQQ: {corr.loc["SPY","QQQ"]:.2f} → both equities, highly correlated")
print(f"GLD vs SPY: {corr.loc["GLD","SPY"]:.2f} → gold is nearly uncorrelated")

---
## 8. Stationarity — Why Quants Model Returns, Not Prices

A **stationary** series has a constant mean and variance over time.

- **Prices** trend up/down → NOT stationary → can't model directly
- **Returns** fluctuate around zero → stationary → safe to model

### Augmented Dickey-Fuller (ADF) test

Tests the null hypothesis: "this series has a unit root (is NOT stationary)."

- **p < 0.05** → reject null → series IS stationary
- **p > 0.05** → cannot reject → series is NOT stationary

If you try to model non-stationary data as stationary, your model produces garbage. This is why every quant function takes **returns**, not prices.

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Test prices
adf_prices = adfuller(prices.dropna(), autolag="AIC")
print(f"ADF test on PRICES:")
print(f"  p-value: {adf_prices[1]:.6f}")
print(f"  Result:  {"Stationary" if adf_prices[1] < 0.05 else "NOT Stationary"}")
print()

# Test returns
adf_returns = adfuller(returns.dropna(), autolag="AIC")
print(f"ADF test on RETURNS:")
print(f"  p-value: {adf_returns[1]:.6f}")
print(f"  Result:  {"Stationary" if adf_returns[1] < 0.05 else "NOT Stationary"}")
print()
print("→ Always model RETURNS, never raw prices.")

---
## Exercises

1. **Change TICKER to AAPL** at the top and re-run. Compare kurtosis to SPY — which has fatter tails?
2. **Compare QQQ vs XLE** — which has higher vol? Higher skew? What does that tell you about tech vs energy?
3. **Rolling correlation:** Run the cell below. Does SPY/TLT correlation spike more negative during the COVID crash?

In [ ]:
# Exercise 3: Rolling correlation
rolling_corr = multi_returns["SPY"].rolling(60).corr(multi_returns["TLT"])
rolling_corr.plot(figsize=(12, 4), title="60-Day Rolling Correlation: SPY vs TLT")
plt.axhline(0, color="grey", linewidth=0.5)
plt.ylabel("Correlation")
plt.show()
print("Look for the dip around March 2020.")
print("When stocks crash, the flight to bonds makes correlation more negative.")